# OCR Models benchmarks

## 1. PaddleOCR

In [ ]:
!pip install paddlepaddle-gpu paddlex

### Downloading MWS Dataset

In [ ]:
import json
from pathlib import Path
from datasets import load_dataset

dataset_mws = load_dataset("MTSAIR/MWS-Vision-Bench")

print(dataset_mws)
print(dataset_mws['train'][0])

DatasetDict({
    train: Dataset({
        features: ['image', 'id', 'type', 'dataset_name', 'question', 'answers'],
        num_rows: 1302
    })
})
{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=905x1280 at 0x79E52DACBB90>, 'id': '1', 'type': 'text grounding ru', 'dataset_name': 'business', 'question': 'Где находится герб на документе? Выведи абсолютные координаты (в пикселях) левого верхнего и правого нижнего углов ограничивающего прямоугольника (bounding box). Координаты должны быть указаны в пикселях относительно исходного размера изображения 905x1280. Твой ответ должен быть в следующем формате: (x1, y1, x2, y2) # x1, y1, x2, y2 — это абсолютные координаты (в пикселях) ограничивающего прямоугольника (bounding box).', 'answers': ['398', '65', '467', '140']}


### Downloading MERA Dataset

In [ ]:
from datasets import load_dataset

dataset_lab = load_dataset("MERA-evaluation/LabTabVQA")
print(dataset_lab)
print(dataset_lab['shots'][0])

DatasetDict({
    shots: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 10
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 339
    })
})
{'instruction': 'Изображение: <image>\nВопрос:\n{question}\n\nA. {option_a}\nB. {option_b}\nC. {option_c}\nD. {option_d}\nE. {option_e}\nF. {option_f}\nG. {option_g}', 'inputs': {'image': {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x07n\x00\x00\x01|\x08\x02\x00\x00\x00\x88k\x0c\x0b\x00\x01\x00\x00IDATx\x9c\xec\xfdWw\xe3\xd8\xb6\xde\x0f/0\x81\x04\x08\xe6\x9c\x95\xa5\xaa\xea\xdc\xbb}\xce\xf0\xb0/\xfd\xbd\xfe\xfeT\xbe\xf0\x8d\xc7\xb0}\xce\xee\xdd\xfbtW\x95JY\x94(\xe6\x9c\x00f\x12\xef\xc5\xf3j\x19MJ\x94*W\xa9\xe7\xef\xa2G5\x85\xb0\x00,`\xcd\xf5\xac\x19\x04]\xd7\x19A\x10\x04A\x10\x04A\x10\x04A\x10\x04A\x10\x04A\xdc\x8f\xe9s7\x80 \x08\x82 \x08\x82 \x08\x82 \x08\x82 \x08\x82\xf8\xd2!)\x99 \x08\x82 \x08\x82 \x08\x82 \x08\x82 \x08\x82x\x00\x92\

In [ ]:
!pip install langchain

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

model_path = "PaddlePaddle/PaddleOCR-VL-1.5"

model = AutoModelForImageTextToText.from_pretrained(model_path, torch_dtype=torch.bfloat16).to("cuda").eval()
processor = AutoProcessor.from_pretrained(model_path)

`torch_dtype` is deprecated! Use `dtype` instead!
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


Loading weights:   0%|          | 0/608 [00:00<?, ?it/s]

The image processor of type `PaddleOCRVLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


In [ ]:
!pip install -q evaluate jiwer

### Eval on MWS (idk)

In [ ]:
def build_question(item):
    task_type = item["type"]
    question = item["question"]

    if task_type == "full-page OCR ru":
        return "Распознай весь текст на изображении дословно."
    elif task_type == "document parsing ru":
        return "Распознай структуру документа и весь текст."
    else:
        return question

def predict(image, question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image.convert("RGB")},
                {"type": "text", "text": question},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0].strip()

### Removing useless (for Paddle) tasks

In [ ]:
def clean_pred(text: str) -> str:
    """Removing LOC|| artefacts of output"""
    text = re.sub(r"<\|LOC_\d+\|>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
SUPPORTED_TYPES = {"full-page OCR ru", "document parsing ru"}

items = [
    item for item in dataset_mws['train']
    if item["type"] in SUPPORTED_TYPES
]

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

results = []
CHECKPOINT_EVERY = 20
SAVE_PATH = "paddle_results.csv"

start_idx = 0
if Path(SAVE_PATH).exists():
    df_existing = pd.read_csv(SAVE_PATH)
    start_idx = len(df_existing)
    results = df_existing.to_dict("records")
    print(f"Resuming from sample {start_idx}")

results = []
start_idx = 0
if Path(SAVE_PATH).exists():
    df_existing = pd.read_csv(SAVE_PATH)
    start_idx = len(df_existing)
    results = df_existing.to_dict("records")
    print(f"Resuming from {start_idx}")

total = len(items)

for i, item in enumerate(tqdm(items[start_idx:], initial=start_idx, total=total), start=start_idx):
    question = build_question(item)
    pred = clean_pred(predict(item["image"], question))
    gt   = item["answers"][0] if item["answers"] else ""

    results.append({
        "id":           item["id"],
        "type":         item["type"],
        "prediction":   pred,
        "ground_truth": gt,
        "all_answers":  item["answers"],
    })

    print(f"[{i+1}/{total}] {item['type']}")
    print(f"  GT:   {gt[:80]}")
    print(f"  PRED: {pred[:80]}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
        print(f"Checkpoint at {i+1}")

df = pd.DataFrame(results)
df.to_csv(SAVE_PATH, index=False)
print(f"Done: {len(df)} samples")


  0%|          | 0/387 [00:00<?, ?it/s]

[1/387] full-page OCR ru
  GT:   ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И БЛАГОПОЛУЧИЯ ЧЕ
  PRED: ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И<|LOC_165|><|LOC
[2/387] document parsing ru
  GT:   <center>ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И БЛАГОПО
  PRED: ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И<|LOC_165|><|LOC
[3/387] full-page OCR ru
  GT:   Итоговая экономия достигнута по причине проведения конкурсных процедур при подбо
  PRED: Пункт 4.1 бюд.летеня<|LOC_734|><|LOC_18|><|LOC_940|><|LOC_18|><|LOC_940|><|LOC_3
[4/387] document parsing ru
  GT:   **<p align="right"><u>Пункт 4.1 бюллетеня</u></p>**
**<center>Выполнение заплани
  PRED: Пункт 4.1 бюд.летеня<|LOC_736|><|LOC_18|><|LOC_940|><|LOC_18|><|LOC_940|><|LOC_3
[5/387] full-page OCR ru
  GT:   ОБЯЗАННОСТИ ЗАКАЗЧИКА:

2.12. Заказчик обязуется присылат
  PRED: ДОГОВОР ФРАХТОВАНИЯ № 2<|LOC_352|><|LOC_54|><|LOC_645|><|LOC_54|><|LOC_645|><|LO
[6

KeyboardInterrupt: 

import evaluate

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

def token_f1(pred: str, refs: list[str]) -> float:
    pred_tok = set(pred.lower().split())
    best = 0.0
    for ref in refs:
        ref_tok = set(ref.lower().split())
        common = pred_tok & ref_tok
        if not common:
            continue
        p = len(common) / len(pred_tok) if pred_tok else 0
        r = len(common) / len(ref_tok) if ref_tok else 0
        best = max(best, 2 * p * r / (p + r))
    return best

df["f1"] = df.apply(
    lambda row: token_f1(row["prediction"], row["all_answers"]),
    axis=1
)

ocr_types = {"full-page OCR ru", "document parsing ru"}
df_ocr = df[df["type"].isin(ocr_types)]

preds_ocr = df_ocr["prediction"].tolist()
refs_ocr  = df_ocr["ground_truth"].tolist()

cer = cer_metric.compute(predictions=preds_ocr, references=refs_ocr) if preds_ocr else None
wer = wer_metric.compute(predictions=preds_ocr, references=refs_ocr) if preds_ocr else None

print("=== PaddleOCR-VL-1.5 | MWS-Vision-Bench ===")
print(f"Всего семплов:    {len(df)}")
print(f"OCR семплов:      {len(df_ocr)}")
print(f"Token F1 (all):   {df['f1'].mean():.4f}")
print(f"CER (OCR tasks):  {cer:.4f}" if cer else "CER: нет OCR сэмплов")
print(f"WER (OCR tasks):  {wer:.4f}" if wer else "WER: нет OCR сэмплов")

print("\n=== F1 по типам задач ===")
print(df.groupby("type")["f1"].mean().round(4).to_string())


### Eval on MERA

In [ ]:
import re
import torch

OPTIONS = ["A", "B", "C", "D", "E", "F", "G"]

def build_prompt_meta(item):
    inp = item["inputs"]
    q   = inp["question"]
    opts = "\n".join([
        f"A) {inp['option_a']}",
        f"B) {inp['option_b']}",
        f"C) {inp['option_c']}",
        f"D) {inp['option_d']}",
        f"E) {inp['option_e']}",
        f"F) {inp['option_f']}",
        f"G) {inp['option_g']}",
    ])
    return f"{q}\n\n{opts}\n\nОтветь одной буквой: A, B, C, D, E, F или G."

def extract_option(text: str) -> str:
    match = re.search(r"\b([A-G])\b", text.upper())
    return match.group(1) if match else "A"  # fallback

def predict_mc(image, prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image.convert("RGB")},
                {"type": "text",  "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    raw = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
    return extract_option(raw), raw.strip()

In [ ]:
from PIL import Image
import io

def get_image(item) -> Image.Image:
    img = item['inputs']['image']
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    elif isinstance(img, dict):
        if 'bytes' in img and img['bytes']:
            return Image.open(io.BytesIO(img['bytes'])).convert("RGB")
        elif 'path' in img:
            return Image.open(img['path']).convert("RGB")
    raise ValueError(f"Unknown image format: {type(img)}")

In [ ]:
from tqdm.auto import tqdm
from pathlib import Path

SAVE_PATH = "labtabvqa_results.csv"
CHECKPOINT_EVERY = 20

results = []
start_idx = 0
if Path(SAVE_PATH).exists():
    df_e = pd.read_csv(SAVE_PATH)
    start_idx = len(df_e)
    results = df_e.to_dict("records")
    print(f"Resuming from {start_idx}")

split = dataset_lab['test']
items = list(split)
total = len(items)

for i, item in enumerate(tqdm(items[start_idx:], initial=start_idx, total=total), start=start_idx):
    prompt = build_prompt_meta(item)
    pred_letter, raw_output = predict_mc(get_image(item), prompt)
    gt = item["outputs"]  # "A", "B", ... "G"

    results.append({
        "id":         item["meta"]["id"],
        "prediction": pred_letter,
        "ground_truth": gt,
        "raw_output": raw_output,
        "correct":    int(pred_letter == gt),
    })

    print(f"[{i+1}/{total}] GT={gt} PRED={pred_letter} | raw: {raw_output}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
        print(f"Checkpoint at {i+1}")

df = pd.DataFrame(results)
df.to_csv(SAVE_PATH, index=False)

  0%|          | 0/339 [00:00<?, ?it/s]

[1/339] GT= PRED=A | raw: Сокорона кроме
27.07.2
[2/339] GT= PRED=A | raw: ИФА диагностика

Показатель
Рез
[3/339] GT= PRED=A | raw: Диагноз: Z34.0

Нанимен


KeyboardInterrupt: 

## Qwen-2VL

In [ ]:
!pip install qwen-vl-utils

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2-VL-2B-Instruct"

processor_qwen = AutoProcessor.from_pretrained(
    model_id,
    min_pixels=256*28*28,
    max_pixels=512*28*28
)
model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

In [ ]:
def predict_qwen(image, question):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image.convert("RGB")},
                {"type": "text",  "text": question},
            ],
        }
    ]

    text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor_qwen(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model_qwen.device)

    with torch.inference_mode():
        generated_ids = model_qwen.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    return processor_qwen.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0].strip()

def clean_pred(text: str) -> str:
    text = re.sub(r"<\|LOC_\d+\|>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [ ]:
def build_question_qwen(item):
    task_type = item["type"]
    if task_type == "full-page OCR ru":
        return "Распознай весь текст на изображении. Отвечай только на русском языке и в качестве ответа дословно напиши то, что распознал. Ничего не добавляй от себя."
    elif task_type == "document parsing ru":
        return "Распознай всю структуру и текст документа.Отвечай только на русском языке и в качестве ответа дословно напиши то, что распознал. Ничего не добавляй от себя."
    return item["question"] + "\nОтвечай только на русском языке."


In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

SUPPORTED_TYPES = {"full-page OCR ru", "document parsing ru"}
SAVE_PATH = "qwen_mws_results.csv"
CHECKPOINT_EVERY = 20

results = []
start_idx = 0
if Path(SAVE_PATH).exists():
    df_e = pd.read_csv(SAVE_PATH)
    start_idx = len(df_e)
    results = df_e.to_dict("records")
    print(f"Resuming from {start_idx}")

items = [x for x in dataset_mws['train'] if x["type"] in SUPPORTED_TYPES]
total = len(items)
print(f"Сэмплов: {total}")

for i, item in enumerate(tqdm(items[start_idx:], initial=start_idx, total=total), start=start_idx):
    question = build_question_qwen(item)
    pred = clean_pred(predict_qwen(item["image"], question))
    gt = item["answers"][0] if item["answers"] else ""

    results.append({
        "id":           item["id"],
        "type":         item["type"],
        "prediction":   pred,
        "ground_truth": gt,
        "all_answers":  item["answers"],
    })

    print(f"[{i+1}/{total}] {item['type']}")
    print(f"  GT:   {gt[:80]}")
    print(f"  PRED: {pred[:80]}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
        print(f"Checkpoint at {i+1}")

df_qwen = pd.DataFrame(results)
df_qwen.to_csv(SAVE_PATH, index=False)
print(f"Done: {len(df_qwen)} samples")


Сэмплов: 387


  0%|          | 0/387 [00:00<?, ?it/s]

[1/387] full-page OCR ru
  GT:   ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И БЛАГОПОЛУЧИЯ ЧЕ
  PRED: Постановление Челябинска от 14.03.2025 года
[2/387] document parsing ru
  GT:   <center>ФЕДЕРАЛЬНАЯ СЛУЖБА ПО НАДЗОРУ В СФЕРЕ ЗАЩИТЫ ПРАВ ПОТРЕБИТЕЛЕЙ И БЛАГОПО
  PRED: Постановление Челябинска от 14.03.2025 года об усилении мер по обеспечению санит
[3/387] full-page OCR ru
  GT:   Итоговая экономия достигнута по причине проведения конкурсных процедур при подбо
  PRED: Выполнение запланированных работ на 2023 год по текущему ремонту общего имуществ
[4/387] document parsing ru
  GT:   **<p align="right"><u>Пункт 4.1 бюллетеня</u></p>**
**<center>Выполнение заплани
  PRED: План выполнения запланированных работ на 2023 год по текущему ремонту общего иму
[5/387] full-page OCR ru
  GT:   ОБЯЗАННОСТИ ЗАКАЗЧИКА:

2.12. Заказчик обязуется присылат
  PRED: Договор фрахтования № 2 Санкт-Петербург 10.01.2023 ООО в лице Генерального дирек
[6/387] document parsing ru
  GT:   <ce

KeyboardInterrupt: 

In [ ]:
def get_image(item) -> Image.Image:
    img = item["inputs"]["image"]
    if isinstance(img, Image.Image):
        return img.convert("RGB")
    elif isinstance(img, dict):
        if "bytes" in img and img["bytes"]:
            return Image.open(io.BytesIO(img["bytes"])).convert("RGB")
        elif "path" in img:
            return Image.open(img["path"]).convert("RGB")
    raise ValueError(f"Unknown image format: {type(img)}")

def build_prompt(item) -> str:
    inp = item["inputs"]
    options = "\n".join([
        f"A) {inp['option_a']}",
        f"B) {inp['option_b']}",
        f"C) {inp['option_c']}",
        f"D) {inp['option_d']}",
        f"E) {inp['option_e']}",
        f"F) {inp['option_f']}",
        f"G) {inp['option_g']}",
    ])
    return (
        f"{inp['question']}\n\n"
        f"{options}\n\n"
        "Ответь одной буквой: A, B, C, D, E, F или G."
    )

def extract_option(text: str) -> str:
    match = re.search(r"\b([A-G])\b", text.upper())
    return match.group(1) if match else "A"

def predict_qwen_mc(image: Image.Image, prompt: str):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt},
            ],
        }
    ]

    text = processor_qwen.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor_qwen(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model_qwen.device)

    with torch.inference_mode():
        generated_ids = model_qwen.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
        )

    generated_ids_trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    raw = processor_qwen.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0].strip()

    return extract_option(raw), raw

In [ ]:
SAVE_PATH = "qwen_labtabvqa_results.csv"
CHECKPOINT_EVERY = 20

results = []
start_idx = 0
if Path(SAVE_PATH).exists():
    df_e = pd.read_csv(SAVE_PATH)
    start_idx = len(df_e)
    results = df_e.to_dict("records")
    print(f"Resuming from {start_idx}")

items = list(split)
total = len(items)
print(f"Всего сэмплов: {total}")

for i, item in enumerate(tqdm(items[start_idx:], initial=start_idx, total=total), start=start_idx):
    image  = get_image(item)
    prompt = build_prompt(item)
    pred_letter, raw_output = predict_qwen_mc(image, prompt)
    gt = item["outputs"]

    results.append({
        "id":           item["meta"]["id"],
        "prediction":   pred_letter,
        "ground_truth": gt,
        "raw_output":   raw_output,
        "correct":      int(pred_letter == gt) if gt else None,
    })

    print(f"[{i+1}/{total}] GT={gt or '?'} PRED={pred_letter} | raw: {raw_output[:60]}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
        print(f"Checkpoint at {i+1}")

df = pd.DataFrame(results)
df.to_csv(SAVE_PATH, index=False)
print(f"Done: {len(df)} samples → {SAVE_PATH}")

Всего сэмплов: 339


  0%|          | 0/339 [00:00<?, ?it/s]

[1/339] GT=? PRED=A | raw: A) 1333
[2/339] GT=? PRED=A | raw: A
[3/339] GT=? PRED=C | raw: C
[4/339] GT=? PRED=C | raw: C) 2
[5/339] GT=? PRED=A | raw: A
[6/339] GT=? PRED=A | raw: A
[7/339] GT=? PRED=D | raw: D
[8/339] GT=? PRED=A | raw: A
[9/339] GT=? PRED=A | raw: A) -2
[10/339] GT=? PRED=C | raw: C


KeyboardInterrupt: 

## LightON 0.9B

In [ ]:
!pip install vllm pypdfium2

In [ ]:
import subprocess, time, requests

server = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "lightonai/LightOnOCR-0.9B-32k-1025",
        "--dtype", "bfloat16",
        "--max-model-len", "8192",
        "--port", "8000",
        "--gpu-memory-utilization", "0.90",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Ждём до 3 минут
for i in range(36):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print("✅ Server ready!")
            break
    except:
        print(f"Waiting... {(i+1)*5}s")
else:
    print("❌ Failed. Logs:")
    print(server.stdout.read(10000).decode())


Waiting... 5s
Waiting... 10s
Waiting... 15s
Waiting... 20s
Waiting... 25s
Waiting... 30s
Waiting... 35s
Waiting... 40s
Waiting... 45s
Waiting... 50s
Waiting... 55s
Waiting... 60s
Waiting... 65s
Waiting... 70s
Waiting... 75s
Waiting... 80s
Waiting... 85s
Waiting... 90s
Waiting... 95s
Waiting... 100s
Waiting... 105s
Waiting... 110s
Waiting... 115s
Waiting... 120s
Waiting... 125s
Waiting... 130s
Waiting... 135s
Waiting... 140s
Waiting... 145s
Waiting... 150s
Waiting... 155s
Waiting... 160s
Waiting... 165s
Waiting... 170s
Waiting... 175s
Waiting... 180s
❌ Failed. Logs:
2026-02-18 17:26:59.451822: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771435619.475267   27938 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771435619.482362   27938 cuda_

In [ ]:
import requests

try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f"Status: {r.status_code}")  # 200 = готов
except Exception as e:
    print(f"Не готов: {e}")


Не готов: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a5bf9e9d2b0>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import base64, requests, io, re
from PIL import Image

ENDPOINT = "http://localhost:8000/v1/chat/completions"
MODEL = "lightonai/LightOnOCR-0.9B-32k-1025"

def predict_lighton(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="PNG")
    image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    payload = {
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": [{
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_b64}"}
            }]
        }],
        "max_tokens": 1024,
        "temperature": 0.0,
    }

    response = requests.post(ENDPOINT, json=payload, timeout=120)
    return response.json()["choices"][0]["message"]["content"]

def clean_pred(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_gt(text):
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\*\*|\*|__|_", "", text)
    return re.sub(r"\s+", " ", text).strip()


In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset

dataset_mws = load_dataset("MTSAIR/MWS-Vision-Bench")
SUPPORTED_TYPES = {"full-page OCR ru", "document parsing ru"}
SAVE_PATH = "lighton_mws_results.csv"
CHECKPOINT_EVERY = 20

results, start_idx = [], 0
if Path(SAVE_PATH).exists():
    df_e = pd.read_csv(SAVE_PATH)
    start_idx = len(df_e)
    results = df_e.to_dict("records")
    print(f"Resuming from {start_idx}")

items = [x for x in dataset_mws["train"] if x["type"] in SUPPORTED_TYPES]
total = len(items)
print(f"Сэмплов: {total}")

for i, item in enumerate(tqdm(items[start_idx:], initial=start_idx, total=total), start=start_idx):
    pred = clean_pred(predict_lighton(item["image"]))
    gt   = clean_gt(item["answers"][0] if item["answers"] else "")

    results.append({
        "id":         item["id"],
        "type":       item["type"],
        "prediction": pred,
        "ground_truth": gt,
    })

    print(f"[{i+1}/{total}] {item['type']}")
    print(f"  GT:   {gt[:80]}")
    print(f"  PRED: {pred[:80]}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(SAVE_PATH, index=False)
        print(f"Checkpoint at {i+1}")

df = pd.DataFrame(results)
df.to_csv(SAVE_PATH, index=False)
print(f"Done: {len(df)} samples")


Сэмплов: 387


  0%|          | 0/387 [00:00<?, ?it/s]

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a5bf9565310>: Failed to establish a new connection: [Errno 111] Connection refused'))